<a href="https://colab.research.google.com/github/mariammoatazmokhtar-crypto/MARIE_Calculator/blob/main/Traffic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
files.upload()

Saving 5- Traffic.rar to 5- Traffic (1).rar
Buffered data was truncated after reaching the output size limit.

In [ ]:
!apt-get install unrar -q
!unrar x "5- Traffic.rar" "/content/"

Reading package lists...
Building dependency tree...
Reading state information...
unrar is already the newest version (1:6.1.5-1ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.

UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from 5- Traffic.rar

Creating    /content/5- Traffic                                       OK
Creating    /content/5- Traffic/test                                  OK
Extracting  /content/5- Traffic/test/predictions.csv                       0%  OK 
Creating    /content/5- Traffic/test/test_images                      OK
Extracting  /content/5- Traffic/test/test_images/img_1.jpg                 0%  OK 
Extracting  /content/5- Traffic/test/test_images/img_10.jpg                0%  OK 
Extracting  /content/5- Traffic/test/test_images/img_100.jpg               0%  OK 
Extracting  /content/5- Traffic/test/test_images/img_101.jpg               0%  OK 
Extracting

In [ ]:
import os
print("🔍 محتويات /content:")
for item in os.listdir("/content/"):
    print(" -", item)

print("\n🐱‍🏍 محتويات مجلد 4-  (لو موجود):")
if os.path.exists("/content/5- Traffic"):
    print(os.listdir("/content/5- Traffic"))
else:
    print("❌ مجلد 4-  مش موجود")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import cv2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import layers, models



# ========== 2. المسارات ==========
DATA_DIR = "/content/5- Traffic/train"
TEST_IMAGES_DIR = "/content/5- Traffic/test"
OUTPUT_CSV_PATH = "/content/predictions.csv"

# ========== 3. الثوابت ==========
SEED = 20
IMG_SIZE = (128, 128)
BATCH_SIZE = 16
EPOCHS = 40
LR = 0.001

np.random.seed(SEED)
tf.random.set_seed(SEED)
def preprocess_edges(img):
    # تحويل لـ grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    # كشف الحواف باستخدام Canny
    edges = cv2.Canny(gray, 50, 150)

    # تحويل الحواف لـ 3 قنوات (عشان تدخل للنموذج)
    edges_3channel = np.stack([edges, edges, edges], axis=2)

    # دمج الصورة الأصلية مع الحواف (ratio 70% original, 30% edges)
    combined = img * 0.3 + edges_3channel * 0.7
    return combined.astype(np.uint8)
    def ensure_rgb(img):
    if img.shape[-1] == 4:
        img = img[..., :3]
    return img


# ========== 4. Data Generator ==========
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.5,
    brightness_range=[0.3, 1.7],
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.3
)

train_ds = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    seed=SEED,
    color_mode='rgb'
)

val_ds = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    seed=SEED,
    color_mode='rgb'
)

num_classes = len(train_ds.class_indices)
print(f"✅ Number of classes: {num_classes}")

# ========== 5. النموذج (MobileNetV2) ==========
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(128, 128, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax')
])

opt = tf.keras.optimizers.Adam(learning_rate=LR)
model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# ========== 6. Early Stopping ==========
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True
)

# ========== 7. التدريب ==========
print("\n🚀 Starting Training...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stop],
    verbose=1
)

# ========== 8. رسم النتائج ==========
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['loss'], label='Train Loss')
ax1.plot(history.history['val_loss'], label='Val Loss')
ax1.set_title('Model Loss')
ax1.legend()

ax2.plot(history.history['accuracy'], label='Train Acc')
ax2.plot(history.history['val_accuracy'], label='Val Acc')
ax2.set_title('Model Accuracy')
ax2.legend()
plt.show()

# ========== 9. التنبؤ على صور test ==========
test_images = [f for f in os.listdir(TEST_IMAGES_DIR) if f.endswith(('.jpg', '.png', '.jpeg'))]
print(f"📸 Test images found: {len(test_images)}")

predicted_labels = []
for img_name in test_images:
    img_path = os.path.join(TEST_IMAGES_DIR, img_name)
    img = tf.keras.utils.load_img(img_path, target_size=IMG_SIZE)
    img_array = tf.keras.utils.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    pred = model.predict(img_array, verbose=0)
    predicted_labels.append(np.argmax(pred))

submission_df = pd.DataFrame({'id': test_images, 'label': predicted_labels})
submission_df.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"✅ File saved to {OUTPUT_CSV_PATH}")

FileNotFoundError: [Errno 2] No such file or directory: '/content/5- Traffic/train'